# 🌾 Precision Farming — Nostradamus IoTO SDK

Multi-zone crop monitoring: each field reports soil moisture, soil and ambient temperature, humidity, light, NDVI, leaf wetness, wind and rainfall, plus the sensor battery level.

This notebook mirrors `examples/agriculture/precision_farming.py` using only `nostradamus_ioto_sdk.NostradamusClient`.
It sends **4 zones × 24 hourly readings** (96 records) to the collection
`precision_farming_data`, then queries, aggregates and deletes part of it.

## Workflow

1. Create the `precision_farming_data` collection with a schema
2. Generate and send 4 zones × 24 hourly readings
3. Query by zone, drought stress, disease risk and NDVI ranking
4. Aggregate soil moisture, ambient temperature and battery statistics
5. Delete a 12-hour window for `ZONE_A_FIELD_01`
6. Delete the collection

## What this notebook highlights

- Drought stress detection (low moisture + high temperature)
- Disease risk detection (high leaf wetness + high humidity)
- NDVI ranking across zones

## Prerequisites

```bash
pip install -e ../..          # from the repository root
pip install python-dotenv
```

| Variable | Purpose |
|----------|---------|
| `NOSTRADAMUS_PROJECT_ID` | Target project UUID |
| `NOSTRADAMUS_MASTER_KEY` | Master key (create/delete collection, delete data) |
| `NOSTRADAMUS_WRITE_KEY` | Write key (send data) |
| `NOSTRADAMUS_READ_KEY` | Read key (query data and statistics) |
| `NOSTRADAMUS_BASE_URL` | Optional API base URL override |

> The notebook is idempotent: if the collection already exists the SDK raises
> `APIError` with status `409`, and Step 1 falls back to reusing it.

In [ ]:
%pip install -q -e ../.. python-dotenv

## Configuration

Each permission level gets its own `NostradamusClient` instance — the SDK injects
the `X-API-Key` header, so there is no manual header handling.

In [ ]:
import os
import random
from collections import Counter
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv

from nostradamus_ioto_sdk import NostradamusClient, StatOperation
from nostradamus_ioto_sdk.exceptions import APIError

load_dotenv()

BASE_URL = os.getenv("NOSTRADAMUS_BASE_URL", "https://nostradamus-ioto.issel.ee.auth.gr")
PROJECT_ID = os.getenv("NOSTRADAMUS_PROJECT_ID", "")
MASTER_KEY = os.getenv("NOSTRADAMUS_MASTER_KEY", "")
WRITE_KEY = os.getenv("NOSTRADAMUS_WRITE_KEY", "")
READ_KEY = os.getenv("NOSTRADAMUS_READ_KEY", "")

_missing = [
    name
    for name, value in {
    "PROJECT_ID": PROJECT_ID,
    "MASTER_KEY": MASTER_KEY,
    "WRITE_KEY": WRITE_KEY,
    "READ_KEY": READ_KEY,
    }.items()
    if not value
]
if _missing:
    raise RuntimeError(
        "Missing required environment variables: " + ", ".join(_missing)
    )

print("🌾 Nostradamus IoTO SDK — Precision Farming")
print(f"base_url={BASE_URL} project_id={PROJECT_ID}")

## Step 1: Create the zone collection

`collections.create()` returns a `CollectionResponse`, so the collection ID comes
straight from the model. The schema declares one sample record and is copied
verbatim from `examples/agriculture/precision_farming.py`.

In [ ]:
COLLECTION_NAME = "precision_farming_data"

COLLECTION_SCHEMA = {
    "key": "ZONE_A_FIELD_01",
    "timestamp": "2026-02-13T10:00:00Z",
    "soil_moisture_pct": 45.5,
    "soil_temperature_c": 22.1,
    "ambient_temperature_c": 25.0,
    "ambient_humidity_pct": 65.0,
    "light_intensity_lux": 85000,
    "ndvi": 0.75,
    "leaf_wetness_duration_min": 30,
    "wind_speed_kmh": 15.5,
    "rainfall_mm": 0.0,
    "battery_pct": 95.0,
}

with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    try:
        collection = master_client.collections.create(
            project_id=PROJECT_ID,
            name=COLLECTION_NAME,
            description="Multi-zone crop monitoring data.",
            tags=["agriculture", "precision-farming", "sensor"],
            collection_schema=COLLECTION_SCHEMA,
        )
        print("✅ Collection created")
    except APIError as exc:
        if exc.status_code != 409:
            raise
        collection = next(
            (
                coll
                for coll in master_client.collections.list(PROJECT_ID)
                if coll.collection_name == COLLECTION_NAME
            ),
            None,
        )
        if collection is None:
            raise
        print("⚠️  Collection already exists — reusing it")

collection_id = str(collection.collection_id)
print(f"collection_id={collection_id}")
collection.model_dump()

## Step 2: Generate and send crop-zone data

`data.send()` accepts a single dict or a list of dicts — the SDK serialises and
authenticates the batch with the write key. `generate_data()` is the
same generator used by the script.

In [ ]:
ZONES = [
    "ZONE_A_FIELD_01",
    "ZONE_B_FIELD_02",
    "ZONE_C_FIELD_03",
    "ZONE_D_FIELD_04",
]

def generate_data(zones, num_readings=24):
    """Generates synthetic precision farming sensor data."""
    data = []
    base_time = datetime.now() - timedelta(hours=num_readings)

    for zone in zones:
        for i in range(num_readings):
            ts = base_time + timedelta(hours=i)

            # Generate realistic random data within specified ranges
            soil_moisture = round(random.uniform(15.0, 65.0), 1)
            soil_temp = round(random.uniform(8.0, 32.0), 1)
            ambient_temp = round(random.uniform(5.0, 38.0), 1)
            ambient_humidity = round(random.uniform(30.0, 85.0), 1)
            light_intensity = random.randint(0, 120000)
            ndvi = round(random.uniform(0.2, 0.9), 2)
            leaf_wetness = random.randint(0, 180)
            wind_speed = round(random.uniform(0.0, 45.0), 1)
            rainfall = round(random.uniform(0.0, 25.0), 1)
            battery = round(random.uniform(60.0, 100.0), 1)

            data.append(
                {
                    "key": zone,
                    "timestamp": ts.isoformat() + "Z",
                    "soil_moisture_pct": soil_moisture,
                    "soil_temperature_c": soil_temp,
                    "ambient_temperature_c": ambient_temp,
                    "ambient_humidity_pct": ambient_humidity,
                    "light_intensity_lux": light_intensity,
                    "ndvi": ndvi,
                    "leaf_wetness_duration_min": leaf_wetness,
                    "wind_speed_kmh": wind_speed,
                    "rainfall_mm": rainfall,
                    "battery_pct": battery,
                }
            )
    return data

data = generate_data(ZONES)
print(f"Generated {len(data)} records")

with NostradamusClient(api_key=WRITE_KEY, base_url=BASE_URL) as write_client:
    write_client.data.send(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        data=data,
    )
print(f"✅ Sent {len(data)} records to {COLLECTION_NAME}")

## Step 3: Query data with domain filters

`data.get()` returns a `DataListResponse` with `.data` (records) and
`.total_count`. Filters are plain dicts combined as an AND. The queries below
mirror the alert conditions from the script:

- **🔍 Zone A readings (limit 5)**
- **🏜️ Drought stress (moisture < 25 % and temp > 30 °C)**
- **🍄 Disease risk (leaf wetness > 120 min and humidity > 80 %)**
- **Top 3 healthiest readings (NDVI)**

In [ ]:
def summary(result, label):
    """Print a compact view of a DataListResponse."""
    print(f"{label}: {len(result.data)} of {result.total_count} record(s)")
    if result.data:
        print("  example:", result.data[0])


with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    # All records, grouped per key
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
    summary(result, "📥 All records")
    counts = Counter(row["key"] for row in result.data)
    print("   per key:", dict(sorted(counts.items())))

    # 🔍 Zone A readings (limit 5)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "key", "operator": "eq", "property_value": "ZONE_A_FIELD_01"},
        ],
        limit=5,
    )
    summary(result, "🔍 Zone A readings (limit 5)")

    # 🏜️ Drought stress (moisture < 25 % and temp > 30 °C)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "soil_moisture_pct", "operator": "lt", "property_value": 25.0},
            {"property_name": "ambient_temperature_c", "operator": "gt", "property_value": 30.0},
        ],
        limit=10,
    )
    summary(result, "🏜️ Drought stress (moisture < 25 % and temp > 30 °C)")

    # 🍄 Disease risk (leaf wetness > 120 min and humidity > 80 %)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        filters=[
            {"property_name": "leaf_wetness_duration_min", "operator": "gt", "property_value": 120},
            {"property_name": "ambient_humidity_pct", "operator": "gt", "property_value": 80.0},
        ],
        limit=10,
    )
    summary(result, "🍄 Disease risk (leaf wetness > 120 min and humidity > 80 %)")

    # Top 3 healthiest readings (NDVI)
    result = read_client.data.get(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        attributes=["key", "ndvi", "timestamp"],
        order_by='{"field": "ndvi", "order": "desc"}',
        limit=3,
    )
    print("🏆 Top 3 healthiest readings (NDVI)")
    for row in result.data:
        print("   >", row)

## Step 4: Statistics

`data.statistics()` takes `stat`, `attribute` and optional `group_by` /
`interval` / `limit`. Use the `StatOperation` enum instead of raw strings.

In [ ]:
with NostradamusClient(api_key=READ_KEY, base_url=BASE_URL) as read_client:
    for label, attribute, stat in [
        ("Average soil moisture (%)", "soil_moisture_pct", StatOperation.AVG),
        ("Maximum ambient temperature (°C)", "ambient_temperature_c", StatOperation.MAX),
        ("Minimum battery level (%)", "battery_pct", StatOperation.MIN),
        ("Distinct zones", "key", StatOperation.DISTINCT),
    ]:
        result = read_client.data.statistics(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            stat=stat,
            attribute=attribute,
        )
        print(f"📈 {label}: {result.get('value', result)}")

## Step 5: Delete a 12-hour window for one zone

`data.delete()` accepts `key`, `timestamp_from` and `timestamp_to` (at least one
required). Here we remove the first half of the 24-hour window for
`ZONE_A_FIELD_01` and compare the record count before and after.

In [ ]:
timestamp_from = (datetime.now(timezone.utc) - timedelta(hours=24)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)
timestamp_to = (datetime.now(timezone.utc) - timedelta(hours=12)).strftime(
    "%Y-%m-%dT%H:%M:%SZ"
)

DELETE_KEY = "ZONE_A_FIELD_01"


with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:

    def count_for(key):
        result = master_client.data.get(
            project_id=PROJECT_ID,
            collection_id=collection_id,
            filters=[{"property_name": "key", "operator": "eq", "property_value": key}],
        )
        return result.total_count

    before = count_for(DELETE_KEY)
    result = master_client.data.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
        key=DELETE_KEY,
        timestamp_from=timestamp_from,
        timestamp_to=timestamp_to,
    )
    after = count_for(DELETE_KEY)

print(f"🗑️  {result.message} (criteria={result.criteria})")
print(f"   {DELETE_KEY}: {before} → {after} record(s)")

## Step 6: Clean up

Remove the demo collection so the notebook can be re-run from scratch.

In [ ]:
with NostradamusClient(api_key=MASTER_KEY, base_url=BASE_URL) as master_client:
    master_client.collections.delete(
        project_id=PROJECT_ID,
        collection_id=collection_id,
    )
print(f"🧹 Collection {collection_id} deleted")
print("✨ Demo complete!")